In [1]:
import pandas as pd
import anndata as ad

In [2]:
path_to_your_data = "./data/salmon.merged.gene_counts.tsv"
path_to_your_metadata = "./meta/meta.xlsx"

gene_id_col = 'gene_id'
condition_columns = ['treatment','Age'] # can be one colum, but you can combine multiple columns for your contrast. It will be merged with "_"
filtering_sum = 10

In [3]:
data = pd.read_csv(path_to_your_data,sep='\t')
data = data.set_index(gene_id_col)

In [4]:
col_for_data = data.columns.tolist()[1]

In [5]:
data = data.loc[data.loc[:,col_for_data:].sum(axis=1)>filtering_sum,:]

In [6]:
data

,gene_name,OC1,OC2,OC3,OC4,OT1,OT2,OT3,OT4,YC1,YC2,YC3,YC4,YT1,YT2,YT3,YT4
gene_id,,,,,,,,,,,,,,,,,
ENSMUSG00000000001,Gnai3,6591.000,6228.000,4984.000,6582.000,7545.000,6983.000,6870.0,7780.000,7062.995,5829.000,5217.997,7280.000,7015.000,7041.000,7321.972,6604.000
ENSMUSG00000000028,Cdc45,2572.000,2436.000,2521.000,2567.000,2445.000,2580.001,2444.0,2417.000,3100.000,3190.999,2437.000,2778.000,2790.000,2914.000,3165.001,2805.000
ENSMUSG00000000031,H19,3.000,3.000,3.000,2.000,5.000,5.000,2.0,3.000,2.000,2.000,5.000,2.000,5.000,8.000,5.000,2.000
ENSMUSG00000000037,Scml2,505.000,519.000,346.000,471.001,401.001,380.001,534.0,365.000,673.000,784.999,551.000,605.000,529.999,461.000,594.000,463.827
ENSMUSG00000000056,Narf,2792.921,2750.001,3790.000,3104.000,2349.000,2179.000,2723.0,2013.000,3596.000,4218.997,2655.000,2585.000,2669.000,2528.000,2821.000,2542.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00002076556,Gm56424,17.467,16.946,19.698,16.771,18.002,0.000,0.0,15.119,35.982,9.902,36.267,24.415,33.133,0.000,10.873,22.361
ENSMUSG00002076601,Scarna13,3.120,0.000,0.000,0.000,0.000,1.424,0.0,0.000,1.203,1.842,1.109,0.000,2.527,0.000,1.721,0.000
ENSMUSG00002076665,Gm54427,0.000,0.000,1.721,0.840,0.000,0.000,0.0,0.000,0.000,2.397,0.000,0.000,2.834,2.869,1.886,0.000


In [7]:
meta = pd.read_excel(path_to_your_metadata)

In [8]:
meta["condition"] = (
    meta[condition_columns]
    .astype(str)
    .agg("_".join, axis=1)
)

In [9]:
meta = meta.set_index('id')

In [10]:
meta

,treatment,stimulation,Age,condition
id,,,,
OC1,Control,NO,Old,Control_Old
OC2,Control,NO,Old,Control_Old
OC3,Control,NO,Old,Control_Old
OC4,Control,NO,Old,Control_Old
OT1,Trained,NO,Old,Trained_Old
OT2,Trained,NO,Old,Trained_Old
OT3,Trained,NO,Old,Trained_Old
OT4,Trained,NO,Old,Trained_Old
YC1,Control,NO,Young,Control_Young


In [11]:
adata = ad.AnnData(data.loc[:,col_for_data:].T,var=data.loc[:,:'gene_name'],obs=meta)

In [12]:
adata.X = adata.X.astype(int)

In [13]:
adata.write_h5ad("./data/adata_filter_genes.h5ad")